In [4]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
from tqdm import tqdm

# STEP 1: Create a FaceLandmarker object.
base_options = python.BaseOptions(model_asset_path='../face_landmarker_v2_with_blendshapes.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

def draw_landmarks_on_frame(frame, detection_result):
    """
    Draws face landmarks on a single video frame.

    Parameters:
    - frame: numpy.ndarray
        The video frame as a NumPy array.
    - detection_result: FaceLandmarkerResult
        The detection result containing face landmarks.

    Returns:
    - annotated_frame: numpy.ndarray
        The frame with face landmarks drawn.
    """
    annotated_frame = frame.copy()
    for face_landmarks in detection_result.face_landmarks:
        for landmark in face_landmarks:
            # Extract normalized coordinates
            x = int(landmark.x * annotated_frame.shape[1])  # Scale to frame width
            y = int(landmark.y * annotated_frame.shape[0])  # Scale to frame height
            # Draw a small circle at the landmark position
            cv2.circle(annotated_frame, (x, y), radius=2, color=(0, 255, 0), thickness=-1)
    return annotated_frame

import cv2
import mediapipe as mp
from tqdm import tqdm

def process_video(video_path, save=False, start_second=0, end_second=None, use_tqdm=True):
    """
    Processes a video, detects landmarks in each frame within the specified range, 
    and saves the annotated video with frame numbers written on each frame.

    Parameters:
    - video_path: str
        Path to the input video.
    - start_second: int
        Start time in seconds for processing the video.
    - end_second: int or None
        End time in seconds for processing the video. If None, processes until the end of the video.

    Returns:
    - blendshape_results: list
        A list of detected blendshape scores for each frame.
    """
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Convert time to frame numbers
    start_frame = int(start_second * fps)
    end_frame = int(end_second * fps) if end_second is not None else total_frames

    # Ensure valid frame ranges
    start_frame = max(0, start_frame)
    end_frame = min(total_frames, end_frame)

    # Set the video capture to the start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # Create the video writer
    blendshape_results = []    
    frames = []
    detection_results = []
    
    # Create a progress bar
    total_process_frames = end_frame - start_frame
    current_frame = start_frame

#     with tqdm(total=end_frame - start_frame, desc="Processing video", unit="frame") as progress_bar:
    while cap.isOpened() and current_frame < end_frame:
        ret, frame = cap.read()
#         if current_frame % 1000 == 0:
#             print(f"frame {current_frame} on {video_path}")

        frames.append(frame)
        if not ret:
            break

        # Convert the frame to an mp.Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        detection_result = detector.detect(mp_image)
        detection_results.append(detection_result)

        frame_blendshapes = [
            {blendshape.category_name: blendshape.score for blendshape in face_blendshapes}
            for face_blendshapes in detection_result.face_blendshapes
        ]
        blendshape_results.append(frame_blendshapes)

        current_frame += 1
#         progress_bar.update(1)

    cap.release()
    

    if save != False:
        output_path = save
        if save == True:
            if start_second <= 0:
                start_second = 'start'
            if end_second == None:
                end_second = 'end'
            output_path = f"output/{video_path.split('/')[-1].split('.')[0]}_{start_second}_{end_second}_annotated.{video_path.split('/')[-1].split('.')[1]}"
            
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        progress_bar = tqdm(total=len(frames) - start_frame, desc="Writing video", unit="frame") if use_tqdm else None
        for frame, detection_result in frames, detection_results:
            annotated_frame = draw_landmarks_on_frame(frame, detection_result)
            # Add the frame number as text on the frame
            frame_number_text = f"Frame: {current_frame + 1}"
            cv2.putText(annotated_frame, frame_number_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            out.write(annotated_frame)
            if use_tqdm:
                pbar.update(1)
        out.release()
        print(f"Annotated video saved at: {output_path}")
        
    return blendshape_results, start_frame, detection_results

In [2]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def process_videos_and_save(input_folder, output_folder, save=False, start_second=0, end_second=None, use_tqdm=False):
    """
    Process all videos in a folder, create a DataFrame from blendshape_results, and save as CSV.
    Skips videos that already have a corresponding CSV file in the output folder.
    """
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Gather all video files from the input folder
    video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp4', '.mov', '.avi', '.MOV'))]

    def process_single_video(video_file):
        """
        Process a single video file, save blendshape results as a CSV.
        """
        input_path = os.path.join(input_folder, video_file)
        output_csv = os.path.join(output_folder, os.path.splitext(video_file)[0] + "_blendshape.csv")

        # Skip processing if the output file already exists
        if os.path.exists(output_csv):
            print(f"Skipping already processed video: {input_path}")
            return

        print(f"Processing video: {input_path}")

#         try:
        # Call the process_video function
        blendshape_results, _ = process_video(input_path, save, start_second, end_second, use_tqdm)
        # Get the keys from the first non-empty frame in blendshape_results, if available
        keys = next((frame[0].keys() for frame in blendshape_results if frame), [])

        # Create the DataFrame, inserting zeros for empty frames
        pd.DataFrame([
            {key: round(value, 2) if isinstance(value, float) else value for key, value in frame[0].items()} if frame else {key: 0 for key in keys}
            for frame in blendshape_results
        ]).to_csv(output_csv, index=False)

        print(f"Saved blendshape results to: {output_csv}")
#         except Exception as e:
#             print(f"Error processing {video_file}: {e}")
    
    with ProcessPoolExecutor() as executor:
        executor.map(process_single_video, video_files)
    

In [ ]:
# Example usage
input_folder = "Raw_30fps/"
output_folder = "Temp/"
blendshape_results = process_videos_and_save(input_folder, output_folder, save=False, start_second=0, end_second=None, use_tqdm=False)

0.0

In [23]:
for i in range(len(blendshape_results)):
    try:
        blendshape_results[i][0]
    except:
        print(df.iloc[i].sum())

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
